# MoPhones Credit Portfolio Case Study

**Author:** Senior Data Analyst Candidate  
**Date:** 2026-04-19  
**Objective:** Evaluate credit portfolio health, customer experience impact, and data quality risks for MoPhones.


## 1) Introduction

This notebook follows a business-first analysis of MoPhones, a smartphone financing provider. The analysis is designed for executive decision-making, with explicit assumptions and reproducible steps.


## 2) Business Context

MoPhones enables customers to acquire smartphones through installment credit plans. That business model creates three core risk vectors:

1. **Default risk:** unpaid balances and write-offs.
2. **Delinquency risk:** arrears and increasing days past due (DPD), which pressure future losses.
3. **Customer experience risk:** collections actions may improve cash recovery while harming satisfaction (NPS).

**Success criteria** for the business are:
- Stable and healthy portfolio trajectory.
- Strong repayment behavior and controlled delinquency/default.
- Positive customer experience to support renewals and referrals.


In [ ]:
# 3) Data Loading
from pathlib import Path
import json

ROOT = Path('.').resolve()
summary_path = ROOT / 'deliverables' / 'outputs' / 'analysis_summary.json'

with summary_path.open(encoding='utf-8') as f:
    summary = json.load(f)

print('Loaded summary from', summary_path)
print('Snapshots:', len(summary['portfolio_metrics']))
print('Age segment rows:', len(summary['age_segment_metrics']))
print('NPS joined accounts:', summary['nps_summary']['joined_accounts'])


## 4) Data Cleaning

Cleaning and standardization were applied in `scripts/mophones_case_analysis.py`:
- Standardized blank column headers to `unnamed_col`.
- Cast numeric fields safely with fallback to `0.0`.
- Normalized missing values using explicit string-empty checks.
- Used consistent snapshot parsing from file names (`dd-mm-yyyy`).
- Parsed `.xlsx` NPS data using XML references to preserve sparse column alignment.

### Assumptions (Data Cleaning)
1. Empty numeric values are treated as `0` to keep pipeline deterministic.
2. Where account statuses differ across levels, `ACCOUNT_STATUS_L2` is preferred for stability.
3. Files are treated as end-of-quarter/evaluation snapshots.


In [ ]:
# Quick data quality view: top missing fields in latest snapshot
for row in summary['latest_missingness_top10']:
    print(f"{row['column']:<28} missing_rate={row['missing_rate']:.2%} ({row['missing_rows']})")


### Key Insights
- Data has structural missingness in selected operational columns, not only random nulls.
- Presence of an `unnamed_col` in later snapshots indicates schema drift.


## 5) Feature Engineering

Required features were engineered as follows:

- **Age band:** binned from `CUSTOMER_AGE` into `18–25`, `26–35`, `36–45`, `46–55`, `55+`.
- **Income proxy and banding:** requested `avg_monthly_income = total_income/duration` could not be computed because `total_income` and contract `duration` are absent in source files.
  - Therefore, a transparent proxy was used: `weekly_rate * 4.33` as installment burden proxy.
  - Bands were aligned to requested income buckets, with explicit limitation noted.

### Assumptions (Feature Engineering)
1. `CUSTOMER_AGE` is assumed to represent customer age at snapshot date.
2. Installment burden proxy is not equivalent to real income; it is used only for directional segmentation.


## 6) Portfolio Analysis

Selected high-impact metrics:
1. Portfolio at Risk (PAR 30+)
2. Default Rate
3. Repayment Rate
4. Average Days Past Due
5. Active vs Closed Accounts Ratio


In [ ]:
# Portfolio trend table
for row in summary['portfolio_metrics']:
    print(
        row['snapshot_date'],
        f"PAR30={row['par30_rate']:.1%}",
        f"Default={row['default_rate']:.1%}",
        f"Repayment={row['repayment_rate']:.1%}",
        f"Avg DPD={row['avg_days_past_due']:.1f}",
        f"Active/Closed={row['active_to_closed_ratio']:.2f}" if row['active_to_closed_ratio'] is not None else 'Active/Closed=N/A'
    )


In [ ]:
# High-risk age segment in latest snapshot by PAR30+
latest = max(r['snapshot_date'] for r in summary['portfolio_metrics'])
latest_segments = [r for r in summary['age_segment_metrics'] if r['snapshot_date'] == latest]
latest_segments = sorted(latest_segments, key=lambda x: x['par30_rate'], reverse=True)
for r in latest_segments:
    print(r['age_band'], f"n={r['accounts']}", f"PAR30={r['par30_rate']:.1%}", f"Default={r['default_rate']:.1%}")


### Key Insights
- Portfolio risk worsens over time: PAR30+, default rate, and average DPD all rise across snapshots.
- Repayment rate stays broadly flat, suggesting collections are not improving despite risk migration.
- **Highest-risk segment (latest snapshot): `55+`** with materially higher PAR30 and default rates.
- Interpretation caution: age distribution appears unrealistic (very high concentration in `55+`), indicating potential age-field quality issues.


## 7) Customer Experience Analysis (Credit × NPS)

NPS was merged with credit records using `Loan Id` ↔ `LOAN_ID`.


In [ ]:
nps = summary['nps_summary']
print('Joined records:', nps['joined_accounts'])
print('Avg NPS (current, DPD<30):', round(nps['avg_nps_current'], 2))
print('Avg NPS (arrears, DPD>=30):', round(nps['avg_nps_arrears_30_plus'], 2))
print('Avg NPS (default):', round(nps['avg_nps_default'], 2))
print('Avg NPS (non-default):', round(nps['avg_nps_non_default'], 2))


### Key Insights
- There is a clear negative relationship between credit stress and customer sentiment:
  - Current customers show materially higher NPS than arrears/default customers.
- **Core business tension:** aggressive collections may protect near-term cashflow but risk worsening NPS and long-term retention/referrals.
- **Practical recommendation:** implement risk-tiered collections journeys (digital nudges for early arrears, empathetic assisted channels before hard lock actions) and track both cure rate and NPS delta.


## 8) Data Quality Review

Major issues identified:
1. **Schema drift:** unnamed extra column appears in later snapshots.
2. **High missingness:** operational/payment fields have very high null rates.
3. **Demographic reliability concerns:** age distribution appears implausibly skewed.
4. **No transaction-level payments table:** limits causal analysis of repayment behavior and interventions.

### Recommended Data Improvements
1. Add a normalized **payment transactions fact table** (timestamp, amount, channel, reversal flag).
2. Define and enforce a **status dictionary** with controlled `ACCOUNT_STATUS_L1/L2` mappings.
3. Track **collections and product interaction events** (reminders sent, lock/unlock, support contacts) for behavioral modeling.


### Key Insights
- Current data supports directional portfolio monitoring, but not robust root-cause diagnostics.
- Improving event-level data and status governance is essential for policy optimization.


## 9) Final Recommendations

1. **Stabilize risk early:** tighten early-warning controls on DPD migration (7→30 and 30→90).
2. **Segment collections strategy:** protect NPS while improving cure rates through differentiated treatment paths.
3. **Fix data foundation:** prioritize transaction table + status taxonomy + behavioral telemetry within next data sprint.

## Bonus: Simple Target Data Model
- `dim_customers` (customer profile, demographics)
- `fact_accounts` (loan origination and terms)
- `fact_snapshots` (periodic risk/account state)
- `fact_payments` (transaction-level payments)
- `fact_nps` (survey responses and sentiment)
- Relationships: customer 1→many accounts; account 1→many snapshots/payments; account 1→many NPS records.
